# 🦠 COVID-19 Data Analysis Project
## Notebook 4: Interactive Dashboard

**Author:** Ahmed  
**Date:** March 2026

---

### 📌 Objectives
In this notebook, we move from static visualizations (matplotlib/seaborn) to **dynamic, interactive dashboards** using `Plotly Express` and `ipywidgets`.

1. **Global KPI Overview**: Dynamic HTML summary cards.
2. **Choropleth World Map**: Hover over countries to see real data for any metric.
3. **Interactive Bar Chart**: Use a dropdown to switch between Cases, Deaths, and Recoveries.
4. **Interactive Scatter & Bubble Charts**: Compare country performance dynamically.

---

## 📦 Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML

import warnings
warnings.filterwarnings('ignore')

print('✅ Plotly & ipywidgets imported successfully!')

---
## 📂 Step 2 — Load the Data

In [ ]:
# Load cleaned data
df = pd.read_csv('../data/processed/cleaned_covid.csv')

print(f'Data Loaded: {df.shape[0]} Countries ready for analysis.')
df.head()

---
## 📈 Step 3 — Global KPI Summary Cards
These cards summarize the absolute total global numbers.

In [ ]:
# Calculate Global Totals
global_cases = int(df['TotalCases'].sum())
global_deaths = int(df['TotalDeaths'].sum())
global_recovered = int(df['TotalRecovered'].sum())

# Create HTML String with CSS styling
kpi_html = f"""
<div style="display:flex; justify-content:space-around; background-color:#f8f9fa; padding:20px; border-radius:10px; box-shadow: 2px 2px 5px grey;">
    <div style="text-align:center;">
        <h2 style="margin:0; color:#3498db;">🦠 Total Cases</h2>
        <h1 style="margin:0; font-size:40px;">{global_cases:,}</h1>
    </div>
    <div style="text-align:center;">
        <h2 style="margin:0; color:#e74c3c;">💀 Total Deaths</h2>
        <h1 style="margin:0; font-size:40px;">{global_deaths:,}</h1>
    </div>
    <div style="text-align:center;">
        <h2 style="margin:0; color:#2ecc71;">🛡️ Total Recovered</h2>
        <h1 style="margin:0; font-size:40px;">{global_recovered:,}</h1>
    </div>
</div>
"""

display(HTML(kpi_html))

---
## 🗺️ Step 4 — Interactive Choropleth Map
A world map showing metrics by country. We use an interactive dropdown using `ipywidgets` to let you switch metrics.

In [ ]:
def plot_world_map(metric):
    # Configure color scale based on the metric
    color_scale = 'Reds' if 'Death' in metric or metric == 'CaseFatalityRate' else \
                  'Greens' if 'Recovered' in metric or metric == 'RecoveryRate' else \
                  'Blues'
                  
    fig = px.choropleth(
        df,
        locations="iso_alpha",         # ISO Alpha-3 codes
        color=metric,                  # Dynamic metric column
        hover_name="Country",          # Show country name on hover
        color_continuous_scale=color_scale,
        title=f"Global Distribution of {metric}",
        projection="natural earth"     # Prettier map projection
    )
    
    fig.update_layout(margin=dict(l=0, r=0, t=50, b=0), height=600)
    fig.show()

# Create dropdown widget
dropdown_map = widgets.Dropdown(
    options=['TotalCases', 'TotalDeaths', 'TotalRecovered', 'TotalTests', 'CaseFatalityRate', 'RecoveryRate'],
    value='TotalCases',
    description='Select Metric:',
    style={'description_width': 'initial'}
)

# Link widget to function
widgets.interact(plot_world_map, metric=dropdown_map);

---
## 📊 Step 5 — Interactive Top 15 Bar Chart
Which countries are leading in different categories? Choose the metric from the dropdown below.

In [ ]:
def plot_top15_bar(metric):
    top15 = df.nlargest(15, metric).sort_values(metric, ascending=True)
    
    # Select bar color based on metric
    color_hex = '#1f77b4'
    if 'Death' in metric or metric == 'CaseFatalityRate':
        color_hex = '#d62728'
    elif 'Recover' in metric or metric == 'RecoveryRate':
        color_hex = '#2ca02c'

    fig = px.bar(
        top15, 
        x=metric, 
        y='Country', 
        orientation='h',
        text=metric,
        title=f"Top 15 Countries by {metric}",
        color_discrete_sequence=[color_hex]
    )
    
    # Format the labels based on if it's a rate or total count
    if 'Rate' in metric:
        fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
    else:
        fig.update_traces(texttemplate='%{text:.3s}', textposition='outside')
        
    fig.update_layout(height=600, yaxis_title="Country Name")
    fig.show()

# Reusing the same options
dropdown_bar = widgets.Dropdown(
    options=['TotalCases', 'TotalDeaths', 'TotalRecovered', 'TestsPerCase', 'CaseFatalityRate', 'RecoveryRate'],
    value='TotalCases',
    description='Select Metric:',
    style={'description_width': 'initial'}
)

widgets.interact(plot_top15_bar, metric=dropdown_bar);

---
## 🫧 Step 6 — Interactive Bubble Chart: Cases vs. Deaths vs. Recovery Rate
- **X-Axis**: Total Cases (Log Scale)
- **Y-Axis**: Total Deaths (Log Scale)
- **Bubble Size**: Total Recovered
- **Bubble Color**: Continent

Hover over any bubble to see exactly which country it is!

In [ ]:
fig = px.scatter(
    df, 
    x="TotalCases", 
    y="TotalDeaths",
    size="TotalRecovered", 
    color="Continent",
    hover_name="Country", 
    log_x=True, 
    log_y=True, 
    size_max=80,
    title="Global Landscape: Cases vs Deaths (Bubble Size = Recoveries)",
    template="plotly_white"
)

fig.update_layout(height=700)
fig.show()

---
## 🔍 Step 7 — Country Lookup Tool
Type or select a country from the dropdown to instantly view its complete COVID-19 statistical profile.

In [ ]:
countries = sorted(df['Country'].unique().tolist())

def show_country_profile(country):
    country_data = df[df['Country'] == country].iloc[0]
    
    html = f"""
    <div style="border: 2px solid #34495e; border-radius: 10px; padding: 20px; font-family: Arial, sans-serif; background-color: #ecf0f1;">
        <h2 style="color: #2c3e50; border-bottom: 2px solid #bdc3c7; padding-bottom: 10px;">🌍 COVID-19 Profile: <b>{country}</b></h2>
        <table style="width:100%; border-collapse: collapse; font-size: 16px;">
            <tr style="border-bottom: 1px solid #ccc;">
                <td style="padding: 10px;"><b>Continent:</b></td>
                <td style="padding: 10px; color: #2980b9;">{country_data['Continent']}</td>
            </tr>
            <tr style="border-bottom: 1px solid #ccc;">
                <td style="padding: 10px;"><b>Population:</b></td>
                <td style="padding: 10px;">{country_data['Population']:,.0f}</td>
            </tr>
            <tr style="border-bottom: 1px solid #ccc; background-color: #ffeaa7;">
                <td style="padding: 10px;"><b>Total Cases:</b></td>
                <td style="padding: 10px; color: #d35400;">{country_data['TotalCases']:,.0f}</td>
            </tr>
            <tr style="border-bottom: 1px solid #ccc; background-color: #ffcccc;">
                <td style="padding: 10px;"><b>Total Deaths:</b></td>
                <td style="padding: 10px; color: #c0392b;">{country_data['TotalDeaths']:,.0f}</td>
            </tr>
            <tr style="border-bottom: 1px solid #ccc; background-color: #ccffcc;">
                <td style="padding: 10px;"><b>Total Recovered:</b></td>
                <td style="padding: 10px; color: #27ae60;">{country_data['TotalRecovered']:,.0f}</td>
            </tr>
            <tr style="border-bottom: 1px solid #ccc;">
                <td style="padding: 10px;"><b>Case Fatality Rate (CFR):</b></td>
                <td style="padding: 10px; color: #8e44ad;"><b>{country_data['CaseFatalityRate']}%</b></td>
            </tr>
            <tr style="border-bottom: 1px solid #ccc;">
                <td style="padding: 10px;"><b>Recovery Rate:</b></td>
                <td style="padding: 10px; color: #16a085;"><b>{country_data['RecoveryRate']}%</b></td>
            </tr>
            <tr>
                <td style="padding: 10px;"><b>Total Tests per Case:</b></td>
                <td style="padding: 10px;"><b>{country_data['TestsPerCase']}</b></td>
            </tr>
        </table>
    </div>
    """
    display(HTML(html))

dropdown_country = widgets.Dropdown(
    options=countries,
    value='USA',
    description='Select Country:',
    style={'description_width': 'initial'}
)

widgets.interact(show_country_profile, country=dropdown_country);

---
## 🎉 Dashboard Complete!

This notebook provides a fully interactive interface to explore COVID-19 data across 210 countries.  
**Next up: Notebook 5 — Final Executive Report!**